## **Description**
In this programming assignment, you are required to implement a contiguous sequential pattern mining algorithm and apply it on text data to mine potential phrase candidates.

**Input**

The provided input file ("reviews_sample.txt") consists of 10,000 online reviews from Yelp users.  The reviews have been stemmed (to remove the postfix of each word so words with similar semantics can have the same form), and most of the punctuation has been removed.  Therefore, each line is basically a list of strings separated by spaces.  

An example line is provided as below:
```
1| cold cheap beer good bar food good service looking great pittsburgh style fish sandwich place breading light fish plentiful good side home cut fry good grilled chicken salad steak soup day homemade lot special great place lunch bar snack beer
```  

**Output**

You need to implement an algorithm to mine contiguous sequential patterns that are frequent in the input data.  A contiguous sequential pattern is a sequence of items that frequently appears as a consecutive subsequence in a database of many sequences.  For example, if the database is 

```
1| A,B,A,C
2| A,C,A,B,A,B
3| B,A,A,C,D
```

and the minimum support is 2, then patterns like "A,B,A" or "A,C" are both frequent contiguous sequential patterns, while the pattern "A,A" is not a frequent contiguous sequential pattern because in the first two sequences the two A's are not consecutive to each other.  Notice that it is still a frequent sequential pattern though.  

Also notice that multiple appearances of a subsequence in a single sequence record only counts once.  For example, the pattern "A,B" appears 1 time in the first sequence and 2 times in the second, but its support should be calculated as 2, as there are only 2 records containing subsequence "A,B".  

When implementing the algorithm, you could use any programming language you like. We only need your result pattern file, not your source code file.

Please set the relative minimum support to 0.01 and run it on the given text file.  In other words, you need to extract all the frequent contiguous sequential patterns that have an absolute support no smaller than 100.

Please write all the frequent contiguous sequential patterns along with their absolute supports into a text file named "patterns.txt".  Every line corresponds to exactly one pattern you found and should be in the following format:

support:item_1;item_2;item_3

For example, suppose the phrase "parking lot" has an absolute support 133, then the line corresponding to this frequent contiguous sequential pattern in "patterns.txt" should be:

133:parking;lot

Notice that the order does matter in sequential pattern mining.  That is to say,

133:lot;parking

may be graded as incorrect.  

Important Tips
Make sure that you format each line correctly in the output file. For instance, use a semicolon instead of other characters to separate different items in the sequence.

Notice that the order does matter in sequential pattern mining.  That is to say,

133:lot;parking

may be graded as incorrect

even if 

133:parking;lot

is a frequent contiguous sequential pattern.

Notice that Length-1 patterns should also be included.  

In [1]:
from collections import defaultdict
from math import ceil
from pathlib import Path

# Paths and support threshold required by the assignment
DATA_PATH = Path("reviews_sample.txt")
OUTPUT_PATH = Path("patterns.txt")
MIN_SUPPORT_RATIO = 0.01


def load_sequences(file_path: Path):
    """Load each review as a token sequence; remove optional record id before '|' if present."""
    sequences = []
    with file_path.open("r", encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line:
                continue
            if "|" in line:
                line = line.split("|", 1)[1].strip()
            tokens = line.split()
            if tokens:
                sequences.append(tokens)
    return sequences


def count_length_1_support(sequences):
    """Count document-level support: each token contributes at most once per sequence."""
    support = defaultdict(int)
    for seq in sequences:
        for token in set(seq):
            support[(token,)] += 1
    return dict(support)


def generate_candidates(prev_frequents):
    """Join step for contiguous patterns: overlap by k-1 tokens."""
    prev_list = sorted(prev_frequents)
    k = len(prev_list[0])
    candidates = set()

    for p in prev_list:
        for q in prev_list:
            if p[1:] == q[:-1]:
                candidates.add(p + (q[-1],))

    if k > 1:
        prev_set = set(prev_frequents)
        pruned = set()
        for cand in candidates:
            # For contiguous patterns, all contiguous (k)-subpatterns must be frequent.
            ok = True
            for i in range(len(cand) - k + 1):
                if cand[i : i + k] not in prev_set:
                    ok = False
                    break
            if ok:
                pruned.add(cand)
        return pruned

    return candidates


def count_candidate_support(sequences, candidates):
    """Count support once per sequence for each candidate."""
    if not candidates:
        return {}

    cand_len = len(next(iter(candidates)))
    support = defaultdict(int)

    for seq in sequences:
        if len(seq) < cand_len:
            continue
        windows = set(tuple(seq[i : i + cand_len]) for i in range(len(seq) - cand_len + 1))
        for w in windows:
            if w in candidates:
                support[w] += 1

    return dict(support)


def mine_frequent_contiguous_patterns(sequences, min_support_abs):
    """Apriori-style level-wise mining for frequent contiguous sequential patterns."""
    all_frequent = {}

    length_1 = count_length_1_support(sequences)
    frequent_k = {p: s for p, s in length_1.items() if s >= min_support_abs}
    all_frequent.update(frequent_k)

    while frequent_k:
        candidates = generate_candidates(list(frequent_k.keys()))
        cand_support = count_candidate_support(sequences, candidates)
        frequent_k = {p: s for p, s in cand_support.items() if s >= min_support_abs}
        all_frequent.update(frequent_k)

    return all_frequent


def write_patterns(output_path: Path, pattern_support):
    """Write output in required format: support:item_1;item_2;..."""
    ordered = sorted(pattern_support.items(), key=lambda x: (-x[1], x[0]))
    with output_path.open("w", encoding="utf-8") as f:
        for pattern, support in ordered:
            f.write(f"{support}:{';'.join(pattern)}\n")


sequences = load_sequences(DATA_PATH)
min_support_abs = ceil(MIN_SUPPORT_RATIO * len(sequences))
all_frequent_patterns = mine_frequent_contiguous_patterns(sequences, min_support_abs)
write_patterns(OUTPUT_PATH, all_frequent_patterns)

print(f"Total sequences: {len(sequences)}")
print(f"Minimum support (absolute): {min_support_abs}")
print(f"Frequent contiguous patterns mined: {len(all_frequent_patterns)}")
print(f"Output file written: {OUTPUT_PATH.resolve()}")

# Preview first 10 lines of output
with OUTPUT_PATH.open("r", encoding="utf-8") as f:
    for idx, line in enumerate(f):
        if idx == 10:
            break
        print(line.rstrip())

Total sequences: 10000
Minimum support (absolute): 100
Frequent contiguous patterns mined: 1040
Output file written: /Users/vananhduy/Documents/Repository_Git_Hub/SP26/DBM302m/coursera labs/Pattern Discovery in Data Mining Part 02/patterns.txt
4041:place
3868:good
3550:food
3080:great
2942:like
2895:time
2872:get
2773:one
2443:service
2281:would
